# Stroke-level Transformer (inspired by SketchXAINet)

Choose the **Vector classification** kernel and run in order. Train from scratch for **20 epochs**.

A shared one-layer BiLSTM (64 hidden units per direction) encodes each stroke's shape.
A linear layer projects its 128-dimensional output. Add a linear embedding of the stroke's
starting location and a learned stroke-order embedding. A CLS token and four pre-norm
Transformer blocks (width 128, 4 heads, FF width 384) classify the sequence of strokes.

This is an adaptation of [SketchXAINet, Qu et al., CVPR 2023](https://sketchxai.github.io/),
not an exact reproduction or a novel stroke encoder. We use our existing normalized cache,
centered coordinates, a smaller network, random initialization, and our current optimizer.
No extra simplification, stroke truncation, augmentation, or retrieved memory is introduced.

Same 345 classes and sample IDs: 5,000 train / 500 validation / 500 test per class;
exclude sketches longer than 250 points. Baseline historical preprocessing caveats still apply.
AdamW lr 0.0003, weight decay 0.01, dropout 0.1, batch 128, clip 1, seed 42,
5% linear warmup then cosine decay over 20 epochs. The stroke BiLSTM runs in float32;
the Transformer uses BF16 autocast when supported. Different architecture/parameter count
means this comparison tests the overall hierarchical design, not tokenization alone.

W&B logs train/validation metrics. Test evaluation is optional and local only.
Each run writes a new `results/transformer_stroke_level_<timestamp>/` directory.


In [1]:
from pathlib import Path
from datetime import datetime
from contextlib import nullcontext
import copy, gzip, hashlib, json, math, random, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/experiment_345_seed42').is_dir()
             and (p / 'src').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the original vector-sketch-classification folder.')
CACHE = ROOT / 'data/experiment_345_seed42'
DATA = ROOT / 'data/processed_full'
REFERENCE_IDS = ROOT / 'results/lstm_baseline_reference_seed42'
LOCAL_LSTM = ROOT / 'results/bilstm_345_matched_seed42'

CFG = dict(seed=42, epochs=20, batch_size=128, lr=3e-4, weight_decay=0.01,
           warmup_fraction=0.05, grad_clip=1.0,
           dim=128, heads=4, layers=4, ff_dim=384, dropout=0.1, max_length=250,
           representation='stroke_shape_location_order_v1', input_dim=5, optimizer='adamw',
           stroke_hidden=64, max_strokes=250, shape_input_dim=4)
COORDINATE_SCALE = float(json.loads((DATA/'manifest.json').read_text())['normalization']['scale'])
if not math.isfinite(COORDINATE_SCALE) or COORDINATE_SCALE <= 0:
    raise ValueError('Invalid training-fitted coordinate scale.')
CFG['coordinate_scale'] = COORDINATE_SCALE
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Optional mixed precision on CUDA; float32 on CPU. No GradScaler is needed for bfloat16.
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
torch.set_num_threads(4)
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(CFG['seed'])
def autocast_context():
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))
def save_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
    temp.replace(path)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
print('Project:', ROOT)
print('Device:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU (full training will be slow)')
print('PyTorch:', torch.__version__, '| bfloat16:', USE_BF16)

Project: c:\Users\shrua\OneDrive\Desktop\11785 IDL\vector-sketch-classification
Device: NVIDIA GeForce RTX 5060 Laptop GPU
PyTorch: 2.11.0+cu128 | bfloat16: True


c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Connect to Weights & Biases

First time only: the next cell installs `wandb` into this notebook's selected Python kernel
if it is missing. Set your project and optional username/team below, then run the connection cell.
Paste your API key into the **hidden prompt**, not into notebook source. It will not appear in
notebook outputs, saved configs, or notebook snapshots. Use the same project for both models.

`WANDB_MODE='online'` streams metrics to your account. Set `'offline'` only if you intentionally
want local W&B logs to sync later. No training starts in the connection cell.


In [2]:
import importlib.util
import sys
# Reuse the project-local SDK if available; otherwise install into the selected kernel.
wandb_sdk_dir = ROOT / '.local_packages/wandb_sdk'
if wandb_sdk_dir.is_dir() and str(wandb_sdk_dir) not in sys.path:
    sys.path.insert(0,str(wandb_sdk_dir))
if importlib.util.find_spec('wandb') is None:
    get_ipython().run_line_magic('pip', 'install wandb')


In [3]:
import wandb
import getpass
WANDB_PROJECT = 'sketch-classification'
WANDB_ENTITY = None     # Optional: your W&B username or team name.
WANDB_RUN_NAME = None   # Optional: a descriptive name; default is the unique output folder.
WANDB_MODE = 'online'
WANDB_READY = False
if WANDB_MODE == 'online':
    if not wandb.login(key=getpass.getpass('Paste your W&B API key (hidden): ')):
        raise RuntimeError('W&B login failed.')
elif WANDB_MODE != 'offline':
    raise ValueError("Choose 'online' or 'offline'.")
WANDB_READY = True
print('W&B configured:',WANDB_PROJECT,'| mode:',WANDB_MODE)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\shrua\_netrc
wandb: Currently logged in as: sihuilin (sihuilin-carnegie-mellon-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


W&B configured: sketch-classification | mode: online


## 1. Audit the existing splits

This reads IDs/metadata and hashes cache files, including the point arrays, to make the
training inputs traceable. It does not load the full point arrays into RAM. Split counts
must be 5,000 / 500 / 500 per category across all 345 categories. Missing data or mismatches
stop execution instead of silently drawing a new subset. Test inputs are audited here;
test predictions and metrics are computed only in the final-test cell.

In [4]:
def audit_data():
    manifest = read_json(DATA / 'manifest.json')
    if manifest['representation'] != 'stroke-3' or manifest['normalization']['fit_split'] != 'train':
        raise ValueError('Expected training-normalized stroke-3 data.')
    if manifest['archive_version'] != 'full':
        raise ValueError('Expected full archives.')
    categories = sorted(manifest['class_to_index'], key=manifest['class_to_index'].get)
    if len(categories) != 345:
        raise ValueError('Expected 345 categories.')
    source_hash = sha256(DATA / 'manifest.json')
    for split_name in ('train','valid','test'):
        cached_scale = read_json(CACHE/split_name/'metadata.json')['coordinate_scale']
        if float(cached_scale) != COORDINATE_SCALE:
            raise ValueError('Cache coordinate scale differs from manifest.')
    reference_cfg = read_json(REFERENCE_IDS / 'config.json')
    if reference_cfg['data']['categories'] != categories:
        raise ValueError('Category order differs from the earlier last-state run.')
    local_provenance = read_json(LOCAL_LSTM / 'provenance.json')
    audit, split_ids = {}, {}
    for split, per_class in [('train',5000), ('valid',500), ('test',500)]:
        folder = CACHE / split
        meta = read_json(folder / 'metadata.json')
        spec = meta['specification']
        expected = dict(source_manifest_sha256=source_hash, split=split,
                        per_class=per_class, max_length=250, seed=42, categories=categories)
        if any(spec.get(k) != v for k,v in expected.items()) or meta['scale_factor'] != 1.0:
            raise ValueError(f'{split}: cache preprocessing/sampling specification mismatch.')
        with gzip.open(folder / 'sample_ids.txt.gz', 'rt', encoding='utf-8') as f:
            ids = f.read().splitlines()
        if ids != read_json(REFERENCE_IDS / f'{split}_sample_ids.json'):
            raise ValueError(f'{split}: ordered sample IDs differ from the earlier last-state run.')
        ids_hash = hashlib.sha256(''.join(x+'\n' for x in ids).encode()).hexdigest()
        if len(ids) != len(set(ids)) or ids_hash != meta['sample_ids_sha256']:
            raise ValueError(f'{split}: invalid IDs/hash.')
        if split in ('train', 'valid'):
            earlier = local_provenance[split]
            for key in ('specification','samples','points','sample_ids_sha256','scale_factor','coordinate_scale','class_counts'):
                if meta[key] != earlier[key]:
                    raise ValueError(f'{split}: metadata differs from local LSTM: {key}')
        labels = np.load(folder / 'labels.npy', mmap_mode='r')
        offsets = np.load(folder / 'offsets.npy', mmap_mode='r')
        points = np.load(folder / 'points.npy', mmap_mode='r')
        lengths = np.diff(offsets)
        if len(labels) != len(ids) or len(ids) != 345*per_class or len(offsets) != len(ids)+1:
            raise ValueError(f'{split}: sample count mismatch.')
        if not np.issubdtype(labels.dtype, np.integer) or np.any(labels < 0) or np.any(labels >= 345):
            raise ValueError(f'{split}: invalid labels.')
        if not np.all(np.bincount(labels, minlength=345) == per_class):
            raise ValueError(f'{split}: class counts differ.')
        if offsets[0] != 0 or offsets[-1] != len(points) or points.shape != (meta['points'],3):
            raise ValueError(f'{split}: invalid point/offset arrays.')
        if points.dtype != np.float32 or np.any(lengths < 1) or np.any(lengths > 250):
            raise ValueError(f'{split}: invalid dtype/lengths.')
        # Scan in chunks: validate all features without loading a full split into RAM.
        for start in range(0, len(points), 1_000_000):
            chunk = points[start:start+1_000_000]
            if not np.isfinite(chunk).all() or not np.isin(chunk[:,2], [0,1]).all():
                raise ValueError(f'{split}: invalid stroke-3 features.')
        if any(not identifier.startswith(categories[int(label)]+'/full/'+split+'/')
               for identifier,label in zip(ids,labels)):
            raise ValueError(f'{split}: IDs and labels disagree.')
        hashes = {name:sha256(folder/name) for name in
                  ['points.npy','offsets.npy','labels.npy','sample_ids.txt.gz','metadata.json']}
        audit[split] = dict(samples=len(ids), sample_ids_sha256=ids_hash,
                            cache_files_sha256=hashes, metadata=meta)
        split_ids[split] = set(ids)
        print(f'{split}: {len(ids):,} sketches; exact earlier IDs verified')
    if any(split_ids[a] & split_ids[b] for a,b in [('train','valid'),('train','test'),('valid','test')]):
        raise ValueError('Splits overlap.')
    remote_hash = reference_cfg['data']['preprocessing']['manifest_sha256']
    audit['comparison'] = dict(local_lstm_metadata_matches=True, all_last_state_ids_match=True,
        current_manifest_sha256=source_hash, remote_manifest_sha256=remote_hash,
        remote_preprocessing_equivalence_certified=(source_hash == remote_hash))
    print('Local LSTM preprocessing metadata matches. Remote manifest matches:', source_hash == remote_hash)
    return categories, audit

CATEGORIES, DATA_AUDIT = audit_data()

train: 1,725,000 sketches; exact earlier IDs verified
valid: 172,500 sketches; exact earlier IDs verified
test: 172,500 sketches; exact earlier IDs verified
Local LSTM preprocessing metadata matches. Remote manifest matches: False


## 2. Load the same sketches and reconstruct geometry

The cache remains unchanged. Five point features carry normalized deltas, pen state and
centered coordinates into the encoder. Grouping happens inside the model: an end-of-stroke
flag closes the current stroke; the next valid point begins the next stroke. The last valid
point always closes its sketch. Pen-up jumps locate the next stroke but are excluded from
that stroke's local shape coordinates.


In [5]:
def add_centered_coordinates(strokes, lengths, coordinate_scale):
    """[B,T,3] -> [B,T,5]; same geometry reconstruction as the point-level notebook."""
    if strokes.ndim != 3 or strokes.shape[-1] != 3:
        raise ValueError('Expected stroke-3 inputs.')
    b,t,_ = strokes.shape
    lengths = lengths.to(strokes.device)
    if lengths.shape != (b,) or torch.any(lengths < 1) or torch.any(lengths > t):
        raise ValueError('Invalid sketch lengths.')
    valid = torch.arange(t,device=strokes.device)[None,:] < lengths[:,None]
    clean = strokes.masked_fill(~valid[:,:,None],0)
    xy = clean[:,:,:2].cumsum(1) * coordinate_scale
    center = (xy * valid[:,:,None]).sum(1,keepdim=True) / lengths[:,None,None]
    centered = (xy-center).masked_fill(~valid[:,:,None],0)
    return torch.cat([clean,centered],dim=-1)

class CachedSplit:
    def __init__(self, split):
        folder = CACHE / split
        self.points = np.load(folder/'points.npy', mmap_mode='r')
        self.offsets = np.load(folder/'offsets.npy', mmap_mode='r')
        self.labels = np.load(folder/'labels.npy', mmap_mode='r')
        self.lengths = np.diff(self.offsets)
    def __len__(self):
        return len(self.labels)
    def batch(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        lengths = self.lengths[indices]
        steps = np.arange(int(lengths.max()))[None,:]
        valid = steps < lengths[:,None]
        gather = self.offsets[indices,None] + np.minimum(steps, lengths[:,None]-1)
        strokes = np.asarray(self.points[gather]).copy()
        strokes[~valid] = 0
        features = add_centered_coordinates(torch.from_numpy(strokes).to(DEVICE),
                                            torch.from_numpy(lengths.copy()).long().to(DEVICE), COORDINATE_SCALE)
        return (features,
                torch.from_numpy(lengths.copy()).long().to(DEVICE),
                torch.from_numpy(np.asarray(self.labels[indices]).copy()).long().to(DEVICE))
    def batches(self, batch_size, shuffle=False, seed=42):
        order = np.random.default_rng(seed).permutation(len(self)) if shuffle else np.arange(len(self))
        for start in range(0,len(self),batch_size):
            yield self.batch(order[start:start+batch_size])

train_data, valid_data = CachedSplit('train'), CachedSplit('valid')

## 3. Shape + location + order -> stroke tokens -> Transformer

Each stroke's shape is `(x-x_start, y-y_start, draw, end)` at its valid points.
A packed BiLSTM encodes shape independently of stroke location. Its final forward/backward
states are concatenated and projected to model width. A linear layer embeds the centered
starting coordinate; a learned table embeds stroke order (0-based within each sketch).
Sum the three embeddings and apply dropout, then prepend CLS and run the Transformer.
Padding is excluded from both the stroke encoder and attention. All strokes are retained.
The final CLS embedding is also available through `model.embed()` for later retrieval.


In [6]:
class AttentionBlock(nn.Module):
    def __init__(self, dim, heads, ff_dim, dropout):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim,3*dim)
        self.out = nn.Linear(dim,dim)
        self.ff = nn.Sequential(nn.Linear(dim,ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim,dim))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, valid):
        b,t,d = x.shape
        q,k,v = self.qkv(self.norm1(x)).reshape(b,t,3,self.heads,d//self.heads).permute(2,0,3,1,4).unbind(0)
        attended = F.scaled_dot_product_attention(q,k,v,attn_mask=valid[:,None,None,:],
                                                  dropout_p=0.0, is_causal=False)
        x = x + self.drop(self.out(attended.transpose(1,2).reshape(b,t,d)))
        return x + self.drop(self.ff(self.norm2(x)))

def group_strokes(features, lengths):
    """Group valid points into strokes without crossing sketch boundaries."""
    if features.ndim != 3 or features.shape[-1] != 5:
        raise ValueError('Expected [B,T,5] point features.')
    b,t,_ = features.shape
    lengths = lengths.to(features.device)
    if b < 1 or lengths.shape != (b,) or lengths.dtype not in (torch.int32,torch.int64):
        raise ValueError('Invalid batch or lengths.')
    if torch.any(lengths < 1) or torch.any(lengths > t):
        raise ValueError('Lengths must lie in [1,T].')
    valid = torch.arange(t,device=features.device)[None,:] < lengths[:,None]
    starts = torch.ones(b,t,dtype=torch.bool,device=features.device)
    starts[:,1:] = features[:,:-1,2] > 0.5
    starts &= valid
    points = features[valid]
    flat_starts = starts[valid]
    point_index = torch.arange(len(points),device=features.device)
    first = point_index[flat_starts]
    stroke_ids = flat_starts.long().cumsum(0)-1
    ranks = point_index-first[stroke_ids]
    stroke_lengths = torch.bincount(stroke_ids,minlength=len(first))
    sketch_ids = torch.arange(b,device=features.device)[:,None].expand(b,t)[valid][first]
    counts = torch.bincount(sketch_ids,minlength=b)
    offsets = counts.cumsum(0)-counts
    orders = torch.arange(len(first),device=features.device)-offsets[sketch_ids]
    xy = points[:,3:5]
    locations = xy[first]
    local_xy = xy-locations[stroke_ids]
    ends = ranks == stroke_lengths[stroke_ids]-1
    shape = torch.cat([local_xy,(~ends).to(xy.dtype)[:,None],ends.to(xy.dtype)[:,None]],dim=1)
    padded = features.new_zeros(len(first),int(stroke_lengths.max().item()),4)
    padded[stroke_ids,ranks] = shape
    return padded,stroke_lengths,locations,sketch_ids,orders,counts

class StrokeLevelTransformer(nn.Module):
    def __init__(self,dim=128,heads=4,layers=4,ff_dim=384,dropout=0.1,
                 max_length=250,stroke_hidden=64,max_strokes=250):
        super().__init__()
        if dim % heads or max_strokes < max_length:
            raise ValueError('Width must divide heads; allow at least one stroke per point.')
        self.embedding_dim,self.max_length = dim,max_length
        self.max_strokes = max_strokes
        self.shape_lstm = nn.LSTM(4,stroke_hidden,bidirectional=True,batch_first=True)
        self.shape_projection = nn.Linear(2*stroke_hidden,dim)
        self.location_embedding = nn.Linear(2,dim)
        self.order_embedding = nn.Embedding(max_strokes,dim)
        self.cls = nn.Parameter(torch.empty(1,1,dim))
        nn.init.normal_(self.cls,std=0.02)
        self.embedding_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([AttentionBlock(dim,heads,ff_dim,dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
    def forward(self,features,lengths):
        if features.ndim != 3 or not 1 <= features.shape[1] <= self.max_length:
            raise ValueError('Invalid point sequence width.')
        shape,stroke_lengths,locations,sketch_ids,orders,counts = group_strokes(features,lengths)
        # Packed recurrent computation stays float32; outer Transformer autocast is preserved.
        with torch.autocast(device_type=features.device.type,enabled=False):
            packed = nn.utils.rnn.pack_padded_sequence(shape.float(),stroke_lengths.cpu(),
                                                     batch_first=True,enforce_sorted=False)
            _,(hidden,_) = self.shape_lstm(packed)
            shape_embedding = torch.cat([hidden[-2],hidden[-1]],dim=1)
        tokens = (self.shape_projection(shape_embedding)+self.location_embedding(locations)
                  +self.order_embedding(orders))
        b = features.shape[0]
        max_count = int(counts.max().item())
        x = tokens.new_zeros(b,max_count,self.embedding_dim)
        x[sketch_ids,orders] = tokens
        x = self.embedding_dropout(x)
        x = torch.cat([self.cls.expand(b,-1,-1).to(x.dtype),x],dim=1)
        valid = torch.arange(max_count,device=x.device)[None,:] < counts[:,None]
        valid = torch.cat([torch.ones(b,1,dtype=torch.bool,device=x.device),valid],dim=1)
        for block in self.blocks: x = block(x,valid)
        return self.norm(x[:,0])

class SketchClassifier(nn.Module):
    def __init__(self,num_classes=345,**encoder_kwargs):
        super().__init__()
        self.encoder = StrokeLevelTransformer(**encoder_kwargs)
        self.head = nn.Linear(self.encoder.embedding_dim,num_classes)
    def embed(self,features,lengths): return self.encoder(features,lengths)
    def forward(self,features,lengths): return self.head(self.embed(features,lengths))

MODEL_CFG = {k:CFG[k] for k in ['dim','heads','layers','ff_dim','dropout','max_length','stroke_hidden','max_strokes']}
def make_model(num_classes=345):
    return SketchClassifier(num_classes,**MODEL_CFG).to(DEVICE)
model = make_model()
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))


SketchClassifier(
  (encoder): StrokeLevelTransformer(
    (shape_lstm): LSTM(4, 64, batch_first=True, bidirectional=True)
    (shape_projection): Linear(in_features=128, out_features=128, bias=True)
    (location_embedding): Linear(in_features=2, out_features=128, bias=True)
    (order_embedding): Embedding(250, 128)
    (embedding_dropout): Dropout(p=0.1, inplace=False)
    (blocks): ModuleList(
      (0-3): 4 x AttentionBlock(
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (qkv): Linear(in_features=128, out_features=384, bias=True)
        (out): Linear(in_features=128, out_features=128, bias=True)
        (ff): Sequential(
          (0): Linear(in_features=128, out_features=384, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=384, out_features=128, bias=True)
        )
        (drop): Dropout(p=0.1, 

## 4. Quick synthetic checks

Checks shapes, padding invariance, mixed-length batches, boundary lengths, finite gradients,
checkpoint reload and learning on a tiny synthetic batch. This is not a QuickDraw score.
The sanity model is discarded before training.

In [7]:
def model_checks():
    seed_all(7)
    raw = torch.tensor([[[1.,0.,0.],[1.,0.,1.],[4.,2.,0.],[0.,1.,1.]]],device=DEVICE)
    lengths = torch.tensor([4],device=DEVICE)
    features = add_centered_coordinates(raw,lengths,1.0)
    shape,lens,loc,owners,orders,counts = group_strokes(features,lengths)
    assert lens.tolist()==[2,2] and orders.tolist()==[0,1] and counts.tolist()==[2]
    expected = torch.tensor([[[0.,0.,1.,0.],[1.,0.,0.,1.]],
                             [[0.,0.,1.,0.],[0.,1.,0.,1.]]],device=DEVICE)
    torch.testing.assert_close(shape,expected)
    torch.testing.assert_close(loc,features[0,[0,2],3:5])
    moved = features.clone(); moved[:,:,3:5] += torch.tensor([2.,-3.],device=DEVICE)
    moved_group = group_strokes(moved,lengths)
    torch.testing.assert_close(shape,moved_group[0])
    torch.testing.assert_close(loc+torch.tensor([2.,-3.],device=DEVICE),moved_group[2])
    # Batch boundaries, consecutive pen lifts, one-point and one-stroke sketches.
    x = torch.randn(3,9,5,device=DEVICE); x[:,:,2]=0
    x[0,[2,5,8],2]=1; x[1,:5,2]=1
    lengths = torch.tensor([9,5,1],device=DEVICE)
    assert group_strokes(x,lengths)[-1].tolist()==[3,5,1]
    probe = make_model(3).eval()
    with torch.no_grad():
        original = probe(x,lengths)
        assert original.shape==(3,3) and torch.isfinite(original).all()
        torch.testing.assert_close(original,probe(F.pad(x,(0,0,0,6),value=999),lengths),atol=2e-5,rtol=2e-5)
        changed=x.clone();changed[1,5:]=-999;changed[2,1:]=999
        torch.testing.assert_close(original,probe(changed,lengths),atol=2e-5,rtol=2e-5)
        torch.testing.assert_close(original[1:2],probe(x[1:2,:5],lengths[1:2]),atol=2e-5,rtol=2e-5)
        for pen in [0,1]:
            longest=torch.zeros(1,250,5,device=DEVICE);longest[:,:,2]=pen
            assert probe(longest,torch.tensor([250],device=DEVICE)).shape==(1,3)
        for bad in [torch.tensor([0,5,1],device=DEVICE),torch.tensor([10,5,1],device=DEVICE)]:
            try:probe(x,bad)
            except ValueError:pass
            else:raise AssertionError('Invalid lengths accepted')
    import io
    buffer=io.BytesIO();torch.save(probe.state_dict(),buffer);buffer.seek(0)
    restored=make_model(3).eval()
    restored.load_state_dict(torch.load(buffer,map_location=DEVICE,weights_only=True))
    with torch.no_grad():torch.testing.assert_close(original,restored(x,lengths))
    probe.train() # Recurrent CUDA backward requires a training-mode forward.
    for module in probe.modules():
        if isinstance(module,nn.Dropout):module.p=0.0
    labels=torch.arange(3,device=DEVICE)
    optimizer=torch.optim.AdamW(probe.parameters(),lr=0.002)
    before=float(F.cross_entropy(probe(x,lengths),labels).detach())
    for _ in range(40):
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():loss=F.cross_entropy(probe(x,lengths),labels)
        loss.backward()
        assert all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
        nn.utils.clip_grad_norm_(probe.parameters(),1.0);optimizer.step()
    after=float(F.cross_entropy(probe(x,lengths),labels).detach())
    assert after<before*0.3
    assert probe.encoder.shape_lstm.weight_ih_l0.grad.abs().sum()>0
    assert probe.encoder.location_embedding.weight.grad.abs().sum()>0
    assert probe.encoder.order_embedding.weight.grad.abs().sum()>0
    print(f'PASS: grouping, geometry separation, padding, save/load, mixed-precision gradients; loss {before:.4f} -> {after:.4f}')
model_checks()
seed_all(CFG['seed'])
model=make_model()


PASS: grouping, geometry separation, padding, save/load, mixed-precision gradients; loss 1.1652 -> 0.0003


## 5. Training and checkpoint functions

AdamW (learning rate 0.0003, weight decay 0.01), plain cross-entropy, batch 128,
gradient clipping 1.0. Warm up for 5% of optimizer steps, then cosine decay to zero.
Keep all 20 epochs; select `best.pt` by validation top-1 only. Timing includes batching,
transfer and model work, with GPU synchronization at epoch boundaries. Save `last.pt`
after every epoch to retain the full training state.

In [8]:
def synchronize():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

def run_epoch(model, data, optimizer=None, epoch=0, step=0, total_steps=1):
    training = optimizer is not None
    model.train(training)
    warmup = max(1,round(total_steps*CFG['warmup_fraction']))
    sums = torch.zeros(4,device=DEVICE,dtype=torch.float64)
    synchronize(); start = time.perf_counter()
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        batches = data.batches(CFG['batch_size'],shuffle=training,seed=CFG['seed']+epoch)
        for x,lengths,labels in tqdm(batches,total=math.ceil(len(data)/CFG['batch_size']),
                                     desc=f'{"train" if training else "valid/test"} {epoch}',leave=False):
            if training:
                factor = ((step+1)/warmup if step < warmup else
                          0.5*(1+math.cos(math.pi*min(1,(step-warmup)/max(1,total_steps-warmup-1)))))
                for group in optimizer.param_groups: group['lr'] = CFG['lr']*factor
                optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                logits = model(x,lengths)
                loss = F.cross_entropy(logits,labels)
            if not torch.isfinite(loss): raise FloatingPointError('Nonfinite loss.')
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),CFG['grad_clip'],error_if_nonfinite=True)
                optimizer.step(); step += 1
            top = logits.topk(min(5,logits.shape[1]),dim=1).indices
            sums += torch.stack([loss.detach().double()*len(labels),
                                 (top[:,0]==labels).sum().double(),
                                 (top==labels[:,None]).any(1).sum().double(),
                                 torch.tensor(len(labels),device=DEVICE,dtype=torch.float64)])
    synchronize(); elapsed = time.perf_counter()-start
    loss_sum,correct,top5,count = sums.tolist()
    return dict(loss=loss_sum/count,acc=correct/count,top5_acc=top5/count,
                samples=int(count),seconds=elapsed),step

def save_checkpoint(path, payload):
    temp = Path(path).with_suffix('.tmp')
    torch.save(payload,temp); temp.replace(path)

def train_local():
    """Start a new model/optimizer at epoch 1 and log one W&B record per epoch."""
    if 'wandb' not in globals() or not globals().get('WANDB_READY',False):
        raise RuntimeError('Run section 0: Weights & Biases connection before training.')
    seed_all(CFG['seed'])
    model = make_model()
    optimizer = torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    out = ROOT/'results'/('transformer_stroke_level_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    out.mkdir(parents=True,exist_ok=False)
    save_json(out/'config.json',CFG); save_json(out/'data_audit.json',DATA_AUDIT)
    precision='bfloat16 autocast' if USE_BF16 else 'float32'
    provenance=dict(torch=torch.__version__,numpy=np.__version__,device=str(DEVICE),
        gpu=torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
        precision=precision,parameters=sum(p.numel() for p in model.parameters()),
        categories=CATEGORIES,architecture='stroke-bilstm-shape-location-learned-order-transformer',
        representation=CFG['representation'],coordinate_scale=COORDINATE_SCALE,
        test_used_for_selection=False,initialization='fresh random weights, no checkpoint loaded')
    save_json(out/'provenance.json',provenance)
    notebook_path=ROOT/'notebooks/Transformer Stroke Level.ipynb'
    if notebook_path.exists(): (out/'notebook_snapshot.ipynb').write_bytes(notebook_path.read_bytes())
    print('New run, starting at epoch 1. Outputs:',out,flush=True)
    history,step,best=[],0,-1.0
    total_steps=CFG['epochs']*math.ceil(len(train_data)/CFG['batch_size'])
    with wandb.init(project=WANDB_PROJECT,entity=WANDB_ENTITY,mode=WANDB_MODE,
            name=WANDB_RUN_NAME or out.name,group='transformer-tokenization-comparison',
            tags=['transformer','stroke-level','sketchxai-inspired','from-scratch'],dir=str(out),
            config=dict(CFG,model='transformer',precision=precision,
                        parameters=provenance['parameters']),
            settings=wandb.Settings(save_code=False,disable_git=True)) as wb_run:
        wb_run.define_metric('epoch')
        for namespace in ['train/*','valid/*','optimizer/*','runtime/*']:
            wb_run.define_metric(namespace,step_metric='epoch')
        wb_run.summary['local_output_dir']=str(out)
        save_json(out/'wandb_run.json',dict(id=wb_run.id,url=wb_run.url,
            project=WANDB_PROJECT,entity=wb_run.entity,mode=WANDB_MODE))
        for epoch in range(1,CFG['epochs']+1):
            if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
            train_metrics,step=run_epoch(model,train_data,optimizer,epoch,step,total_steps)
            valid_metrics,_=run_epoch(model,valid_data,epoch=epoch)
            peak_memory=torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None
            history.append(dict(epoch=epoch,train=train_metrics,valid=valid_metrics,
                                peak_gpu_memory_bytes=peak_memory))
            improved=valid_metrics['acc']>best
            best=max(best,valid_metrics['acc'])
            payload=dict(epoch=epoch,step=step,best_acc=best,config=CFG,
                model_state=model.state_dict(),optimizer_state=optimizer.state_dict(),history=history,
                wandb_run_id=wb_run.id,
                python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [])
            if improved: save_checkpoint(out/'best.pt',payload)
            save_checkpoint(out/'last.pt',payload)
            save_json(out/'history.json',history)
            log=dict(epoch=epoch,**{'train/loss':train_metrics['loss'],
                'train/accuracy':train_metrics['acc'],'train/top5_accuracy':train_metrics['top5_acc'],
                'valid/loss':valid_metrics['loss'],'valid/accuracy':valid_metrics['acc'],
                'valid/top5_accuracy':valid_metrics['top5_acc'],
                'optimizer/learning_rate':optimizer.param_groups[0]['lr'],
                'runtime/train_seconds':train_metrics['seconds'],
                'runtime/valid_seconds':valid_metrics['seconds']})
            if peak_memory is not None: log['runtime/peak_gpu_memory_mb']=peak_memory/(1024**2)
            wb_run.log(log)
            wb_run.summary['best_valid_accuracy']=best
            wb_run.summary['best_epoch']=max(history,key=lambda r:r['valid']['acc'])['epoch']
            wb_run.summary['completed_epochs']=epoch
            print(f"Epoch {epoch:02d}/{CFG['epochs']} | train {train_metrics['acc']:.2%} | "
                  f"valid {valid_metrics['acc']:.2%} | best {best:.2%}",flush=True)
    return out


## 6. Train from scratch with W&B logging

Run the connection cell in section 0 first, then the setup/audit/model/helper cells above.
This training cell always initializes a new model and optimizer at epoch 1. It creates a new
local results directory and W&B run each time. It does not resume any earlier checkpoint.
Local `best.pt` and `last.pt` are saved every epoch as appropriate. W&B records configuration,
train/validation loss, top-1/top-5 accuracy, learning rate, timing, and best validation epoch.
Accuracy values in W&B are fractions (0.78 = 78%). Checkpoints stay local.


In [9]:
RUN_DIR = train_local()

New run, starting at epoch 1. Outputs: c:\Users\shrua\OneDrive\Desktop\11785 IDL\vector-sketch-classification\results\transformer_stroke_level_20261102_194304_968865


Epoch 01/20 | train 15.30% | valid 37.43% | best 37.43%


Epoch 02/20 | train 44.51% | valid 55.56% | best 55.56%


Epoch 03/20 | train 55.11% | valid 62.12% | best 62.12%


Epoch 04/20 | train 60.44% | valid 65.92% | best 65.92%


Epoch 05/20 | train 63.72% | valid 68.29% | best 68.29%


... (8 outputs left out)

Epoch 14/20 | train 72.75% | valid 74.93% | best 74.93%


Traceback (most recent call last):                            
  File "C:\Users\shrua\AppData\Local\Temp\ipykernel_22048\156032515.py", line 79, in train_local
    train_metrics,step=run_epoch(model,train_data,optimizer,epoch,step,total_steps)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\shrua\AppData\Local\Temp\ipykernel_22048\156032515.py", line 25, in run_epoch
    loss.backward()
  File "c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\torch\_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\torch\autograd\__init__.py", line 381, in backward
    _engine_run_backward(
  File "c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\torch\autograd\graph.py", line 869, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
          

epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
optimizer/learning_rate,███▇▇▇▆▅▅▄▃▂▂▁
runtime/peak_gpu_memory_mb,▄▄▄▄▄▄▁▄█▃▁▄▄█
runtime/train_seconds,▃▅█▄▁▁▁▁▁▁▅▅▆▂
runtime/valid_seconds,▁█▇▁▁▁▁▂▁▁▆▅▅▁
train/accuracy,▁▅▆▆▇▇▇▇██████
train/loss,█▄▃▂▂▂▁▁▁▁▁▁▁▁
train/top5_accuracy,▁▆▇▇▇█████████
valid/accuracy,▁▄▆▆▇▇▇▇██████
valid/loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁
+1,...


KeyboardInterrupt: 

## 7. Learning curves and best-checkpoint validation

In [ ]:
history = read_json(RUN_DIR/'history.json')
epochs = [r['epoch'] for r in history]
fig, axes = plt.subplots(1,2,figsize=(11,4))
for split in ['train','valid']:
    axes[0].plot(epochs,[r[split]['loss'] for r in history],label=split)
    axes[1].plot(epochs,[100*r[split]['acc'] for r in history],label=split)
axes[0].set_ylabel('Cross-entropy'); axes[1].set_ylabel('Top-1 accuracy (%)')
for ax in axes:
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.2)
    ax.set_xticks(list(range(1,max(epochs)+1,2)))
fig.tight_layout(); fig.savefig(RUN_DIR/'curves.png',dpi=160); plt.show()
selected = torch.load(RUN_DIR/'best.pt',map_location='cpu',weights_only=False)
model = make_model(); model.load_state_dict(selected['model_state'])
validation,_ = run_epoch(model,valid_data)
if abs(validation['acc']-selected['best_acc']) > 1e-8:
    raise AssertionError('Reloaded validation accuracy differs from checkpoint selection.')
save_json(RUN_DIR/'metrics_valid.json',dict(**validation,epoch=selected['epoch'],split='valid'))
print('Selected epoch:',selected['epoch'],'| validation:',validation)

## 8. Final held-out test (off by default)

Set `RUN_FINAL_TEST=True` only after architecture/training settings are frozen.
This reloads the validation-selected checkpoint and saves overall top-1/top-5, loss,
per-class accuracy and predictions in source order. Do not tune settings using test results.
Existing metrics are displayed instead of repeating the evaluation.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if (RUN_DIR/'metrics_test.json').exists():
        print('Already evaluated:',read_json(RUN_DIR/'metrics_test.json'))
    else:
        selected = torch.load(RUN_DIR/'best.pt',map_location='cpu',weights_only=False)
        model = make_model(); model.load_state_dict(selected['model_state']); model.eval()
        test_data = CachedSplit('test')
        loss_sum=correct=correct5=count=0
        per_count=np.zeros(345,dtype=np.int64); per_correct=np.zeros(345,dtype=np.int64)
        predictions=[]
        synchronize(); start=time.perf_counter()
        with torch.inference_mode():
            for x,lengths,labels in tqdm(test_data.batches(CFG['batch_size']),
                                         total=math.ceil(len(test_data)/CFG['batch_size'])):
                with autocast_context():
                    logits=model(x,lengths)
                top=logits.topk(5,dim=1).indices
                loss_sum += float(F.cross_entropy(logits.float(),labels,reduction='sum'))
                correct += int((top[:,0]==labels).sum())
                correct5 += int((top==labels[:,None]).any(1).sum()); count += len(labels)
                y=labels.cpu().numpy(); pred=top.cpu().numpy(); predictions.append(pred)
                per_count += np.bincount(y,minlength=345)
                per_correct += np.bincount(y[pred[:,0]==y],minlength=345)
        synchronize()
        metrics=dict(split='test',epoch=selected['epoch'],samples=count,loss=loss_sum/count,
                     acc=correct/count,top5_acc=correct5/count,seconds=time.perf_counter()-start,
                     checkpoint_sha256=sha256(RUN_DIR/'best.pt'),
                     sample_ids_sha256=DATA_AUDIT['test']['sample_ids_sha256'])
        np.savez_compressed(RUN_DIR/'test_predictions.npz',top5=np.concatenate(predictions),
                            labels=np.asarray(test_data.labels))
        save_json(RUN_DIR/'per_class_test.json',[
            dict(category=name,samples=int(per_count[i]),acc=float(per_correct[i]/per_count[i]))
            for i,name in enumerate(CATEGORIES)])
        save_json(RUN_DIR/'metrics_test.json',metrics)
        print(metrics)
else:
    print('Test evaluation is off. Freeze settings, then set RUN_FINAL_TEST=True.')

## 9. Compare with your point-level Transformer

Compare best validation accuracy, parameter count, train time and memory on the same cache.
Both runs use 20 epochs and the same optimizer settings. This adaptation changes the stroke
encoder and order representation as well as token granularity; do not attribute differences
solely to point versus stroke tokens. Cite SketchXAINet for the hierarchical architecture.


In [ ]:
point_runs=[]
for folder in (ROOT/'results').glob('transformer_geometry_wandb_*'):
    if (folder/'history.json').exists(): point_runs.append(folder)
for name,folder in [('stroke-level',RUN_DIR)]+[('point-level',p) for p in sorted(point_runs)[-1:]]:
    rows=read_json(folder/'history.json')
    best=max(rows,key=lambda r:r['valid']['acc'])
    provenance=read_json(folder/'provenance.json')
    print(name,'| best validation:',f"{best['valid']['acc']:.2%}",'| epoch:',best['epoch'],
          '| parameters:',provenance['parameters'],'| completed epochs:',len(rows))
